## Импорт

In [839]:
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from scipy.linalg import eigh as sp_eigh
from scipy.linalg import eigh
import time
from sklearn.neighbors import KNeighborsClassifier

## ДатаГенератор

In [203]:
class DataGenerator:
    
    def __init__(self, 
                 max_samples=200,    
                 min_samples=50,     
                 max_features=10,     
                 min_features=2,       
                 max_classes=5,        
                 min_classes=2,        
                 random_state=None):
        
        self.max_samples = max_samples
        self.min_samples = min_samples
        self.max_features = max_features
        self.min_features = min_features
        self.max_classes = max_classes
        self.min_classes = min_classes
        self.random_state = random_state
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def generate(self, test_size=0.2, as_dataframe=False):
        
        n_samples = np.random.randint(self.min_samples, self.max_samples + 1)
        n_features = np.random.randint(self.min_features, self.max_features + 1)
        n_classes = np.random.randint(self.min_classes, self.max_classes + 1)
        n_informative = min(n_features, n_classes * 2)  # чтобы были информативные признаки
        
        X, y = make_classification(
            n_samples=n_samples,
            n_features=n_features,
            n_informative=min(n_informative, n_features),
            n_redundant=max(0, n_features - n_informative),
            n_clusters_per_class=1,
            n_classes=n_classes,
            random_state=self.random_state
        )

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=self.random_state,
            stratify=y
        )

        if as_dataframe:
            feature_names = [f'feature_{i}' for i in range(n_features)]
            X_train = pd.DataFrame(X_train, columns=feature_names)
            X_test = pd.DataFrame(X_test, columns=feature_names)
            y_train = pd.Series(y_train, name='target')
            y_test = pd.Series(y_test, name='target')

        self.last_params = {
            'n_samples': n_samples,
            'n_features': n_features,
            'n_classes': n_classes,
            'n_informative': n_informative
        }
        
        return X_train, X_test, y_train, y_test

## Вероятностный LDA

#### Дискриминантная функция

$$
\delta_y(x) = \log P(y) - \frac{1}{2} \mu_y^T \Sigma^{-1} \mu_y + x^T \Sigma^{-1} \mu_y
$$

In [625]:
class ProbLDA():

    def fit(self,X,y):
        n_samples, n_features = X.shape
        self.unique_classes, n_classes = np.unique(y, return_counts=True)
        self.priors = n_classes / n_samples
        
        #total_means = X.mean(axis=0)
        self.classes_means = np.array([X[y==cls].mean(axis=0) for cls in self.unique_classes])
        #beetween_classes = classes_means - total_means
        within_classes = np.vstack([np.array(X[y==cls] - self.classes_means[idx]) for idx,cls in enumerate(self.unique_classes)])
        #print(within_classes.shape)
        self.cvr_matrix = 1 / n_samples * within_classes.T @ within_classes #(PxP)
        
        self.inv_cvr_matrix  = np.linalg.pinv(self.cvr_matrix) #np.linalg.inv(self.cvr_matrix)
        self.const = np.array([np.log(self.priors[k]) - 0.5 * self.classes_means[k] @ self.inv_cvr_matrix @ self.classes_means[k]
                             for k in range(len(self.unique_classes))])
        
    """
    def discr_func(self,x):      
        return np.array([np.log(prior) - 0.5 * self.classes_means[idx].T @ self.inv_cvr_matrix @ self.classes_means[idx]
                + x.T @ self.inv_cvr_matrix @ self.classes_means[idx] for idx,prior in enumerate(self.priors)])
    
    def predict(self,X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        probs = X.apply(self.discr_func, axis = 1)
        idxs = probs.apply(lambda x: np.argmax(x))
        predictions = np.take(self.unique_classes, idxs)
        return predictions
    """

    def predict(self,X):
        X = np.asarray(X)

        probs = X @ self.inv_cvr_matrix @ self.classes_means.T
        probs += self.const
        max_idxs = np.argmax(probs, axis=1)
        predictions = np.take(self.unique_classes, max_idxs)

        return predictions

## LDA через критерий Фишера (с Хабра)

#### Отношение Рэлея

$$
J(w) = \frac{w^T S_b w}{w^T S_w w}
$$

In [634]:
class Fisher_LDA:
    def __init__(self, n_components=None):
        self.n_components = n_components

    def fit(self, X, y):
        n_samples, n_features = X.shape
        classes, cls_counts = np.unique(y, return_counts=True)
        priors = cls_counts / n_samples

        X_cls_mean = np.array([X[y == cls].mean(axis=0) for cls in classes])
        between_cls_deviation = X_cls_mean - X.mean(axis=0)
        within_cls_deviation = X - X_cls_mean[y]

        Sb = priors * between_cls_deviation.T @ between_cls_deviation
        Sw = within_cls_deviation.T @ within_cls_deviation / n_samples
        reg = 1e-5 * np.trace(Sw) / n_features  # адаптивная регуляризация (через след для сохранения масштабирования (диагональная м-ца зависит от данныъ, а не просто np.eye))
        Sw_reg = Sw + reg * np.eye(n_features)
        evals, evecs = sp_eigh(Sb, Sw_reg)
        self.dvecs = evecs[:, np.argsort(evals)[::-1]]   

        self.weights = X_cls_mean @ self.dvecs @ self.dvecs.T
        self.bias = np.log(priors) - 0.5 * np.diag(X_cls_mean @ self.weights.T)

        if self.n_components is None:
            self.n_components = min(classes.size - 1, n_features)

    def transform(self, X):
        return X @ self.dvecs[:, : self.n_components]

    def predict(self, X_test):
        scores = X_test @ self.weights.T + self.bias

        return np.argmax(scores, axis=1)

## Тесты LDA (prob, sklearn, habr)
#### Вывод: модели не устойчивы при p>n, но справляются при мультиколлинеарности

In [906]:
dg = DataGenerator(
    max_samples=10000,
    min_samples=1000,
    max_features=20,
    min_features=5,
    max_classes=8,
    min_classes=4,
    random_state=44
)

OLS_LDA(reg=1e-5)

accuracies = []
sk_accuracies = []
habr_accuracies = []
ols_accuracies = []
svd_accuracies = []
svd_knn_accuracies = []
klda_accuracies = []
qda_accuracies = []
sk_qda_accuracies = []

times_lda = []
times_sk = []
times_habr = []
times_ols = []
times_svd = []
times_svd_knn = []
times_klda = []
times_qda = []
times_sk_qda = []

for i in range(10):
    X_train, X_test, y_train, y_test = dg.generate(test_size=0.4)

    # sklearn QDA
    start = time.time()
    sk_qda = QuadraticDiscriminantAnalysis(solver='eigen',shrinkage='auto')
    sk_qda.fit(X_train, y_train)
    sk_qda_y_pred = sk_qda.predict(X_test)
    sk_qda_acc = accuracy_score(sk_qda_y_pred, y_test)
    times_sk_qda.append(time.time() - start)
    
    # ProbQDA
    start = time.time()
    qda = ProbQDA(reg=1e-5, shrinkage=0.15)
    qda.fit(X_train, y_train)
    qda_y_pred = qda.predict(X_test)
    qda_acc = accuracy_score(qda_y_pred, y_test)
    times_qda.append(time.time() - start)
    
    # Kenel LDA
    start = time.time()
    klda = KLDA(reg=1e-5, kernel_func="poly", degree=3)
    klda.fit(X_train, y_train)
    k_y_pred = klda.predict(X_test)
    k_acc = accuracy_score(k_y_pred, y_test)
    times_klda.append(time.time() - start)
    
    # ProbLDA
    start = time.time()
    lda = ProbLDA()
    lda.fit(X_train, y_train)
    y_pred = lda.predict(X_test)
    acc = accuracy_score(y_pred, y_test)
    times_lda.append(time.time() - start)

    # sklearn LDA
    start = time.time()
    sk_lda = LinearDiscriminantAnalysis(solver='eigen',shrinkage='auto')
    sk_lda.fit(X_train, y_train)
    sk_y_pred = sk_lda.predict(X_test)
    sk_acc = accuracy_score(sk_y_pred, y_test)
    times_sk.append(time.time() - start)

    # Fisher LDA
    start = time.time()
    habr_lda = Fisher_LDA()
    habr_lda.fit(X_train, y_train)
    habr_y_pred = habr_lda.predict(X_test)
    habr_acc = accuracy_score(habr_y_pred, y_test)
    times_habr.append(time.time() - start)

    # OLS LDA
    start = time.time()
    ols_lda = OLS_LDA(reg=1e-5, bias=False, svd=True)
    ols_lda.fit(X_train, y_train)
    ols_y_pred = ols_lda.predict(X_test)
    ols_acc = accuracy_score(ols_y_pred, y_test)
    times_ols.append(time.time() - start)

    # LDA_SVD
    start = time.time()
    lda_svd = LDA_SVD(n_components=10)
    lda_svd.fit(X_train, y_train)
    svd_y_pred = lda_svd.predict(X_test)
    svd_acc = accuracy_score(y_test, svd_y_pred)
    times_svd.append(time.time() - start)

    # LDA_SVD + KNN
    start = time.time()
    lda_knn = LDA_SVD(n_components=10)
    new_X_train = lda_knn.fit_transform(X_train, y_train)
    new_X_test = lda_knn.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=30)
    knn.fit(new_X_train, y_train)
    knn_y_pred = knn.predict(new_X_test)
    svd_knn_acc = accuracy_score(y_test, knn_y_pred)
    times_svd_knn.append(time.time() - start)

    accuracies.append(acc)
    sk_accuracies.append(sk_acc)
    habr_accuracies.append(habr_acc)
    ols_accuracies.append(ols_acc)
    svd_accuracies.append(svd_acc)
    svd_knn_accuracies.append(svd_knn_acc)
    klda_accuracies.append(k_acc)
    qda_accuracies.append(qda_acc)
    sk_qda_accuracies.append(sk_qda_acc)

    print(
        f"{i+1}: samples={dg.last_params['n_samples']}, "
        f"features={dg.last_params['n_features']}, "
        f"classes={dg.last_params['n_classes']}, "
        f"acc={acc:.3f}",
        f"sk={sk_acc:.3f}",
        f"habr={habr_acc:.3f}",
        f"ols={ols_acc:.3f}",
        f"svd={svd_acc:.3f}",
        f"svd_knn={svd_knn_acc:.3f}",
        f"kernel={k_acc:.3f}",
        f"qda={qda_acc:.3f}",
        f"sk_qda={sk_qda_acc:.3f}",
        f"time_lda={times_lda[-1]:.4f}s",
        f"time_sk={times_sk[-1]:.4f}s",
        f"time_habr={times_habr[-1]:.4f}s",
        f"time_ols={times_ols[-1]:.4f}s",
        f"time_svd={times_svd[-1]:.4f}s",
        f"time_svd_knn={times_svd_knn[-1]:.4f}s",
        f"time_kernel={times_klda[-1]:.4f}s",
        f"time_qda={times_qda[-1]:.4f}s",
        f"time_sk_qda={times_sk_qda[-1]:.4f}s\n" 
    )

# Финальная статистика
print(f"\nСредняя точность LDA: {np.mean(accuracies):.3f}")
print(f"Средняя точность sklearn: {np.mean(sk_accuracies):.3f}")
print(f"Средняя точность habr: {np.mean(habr_accuracies):.3f}")
print(f"Средняя точность OLS: {np.mean(ols_accuracies):.3f}")
print(f"Средняя точность LDA_SVD: {np.mean(svd_accuracies):.3f}")
print(f"Средняя точность LDA_SVD + KNN: {np.mean(svd_knn_accuracies):.3f}")
print(f"Средняя точность KLDA: {np.mean(klda_accuracies):.3f}")
print(f"Средняя точность QDA (моя): {np.mean(qda_accuracies):.3f}")
print(f"Средняя точность QDA (sklearn): {np.mean(sk_qda_accuracies):.3f}") 

print(f"\nСреднее время LDA: {np.mean(times_lda):.4f}s")
print(f"Среднее время sklearn: {np.mean(times_sk):.4f}s")
print(f"Среднее время habr: {np.mean(times_habr):.4f}s")
print(f"Среднее время OLS: {np.mean(times_ols):.4f}s")
print(f"Среднее время LDA_SVD: {np.mean(times_svd):.4f}s")
print(f"Среднее время LDA_SVD + KNN: {np.mean(times_svd_knn):.4f}s")
print(f"Среднее время KLDA: {np.mean(times_klda):.4f}s")
print(f"Среднее время QDA (моя): {np.mean(times_qda):.4f}s")
print(f"Среднее время QDA (sklearn): {np.mean(times_sk_qda):.4f}s") 

1: samples=4491, features=6, classes=7, acc=0.592 sk=0.590 habr=0.592 ols=0.605 svd=0.590 svd_knn=0.793 kernel=0.366 qda=0.739 sk_qda=0.863 time_lda=0.0019s time_sk=0.0326s time_habr=0.0014s time_ols=0.0030s time_svd=0.0533s time_svd_knn=0.0374s time_kernel=5.5074s time_qda=0.0035s time_sk_qda=0.0103s

2: samples=4971, features=5, classes=8, acc=0.703 sk=0.703 habr=0.703 ols=0.688 svd=0.702 svd_knn=0.861 kernel=0.399 qda=0.740 sk_qda=0.927 time_lda=0.0026s time_sk=0.0466s time_habr=0.0017s time_ols=0.0056s time_svd=0.0489s time_svd_knn=0.0360s time_kernel=8.8416s time_qda=0.0030s time_sk_qda=0.0103s

3: samples=3947, features=20, classes=6, acc=0.688 sk=0.688 habr=0.688 ols=0.668 svd=0.689 svd_knn=0.743 kernel=0.241 qda=0.933 sk_qda=0.987 time_lda=0.0024s time_sk=0.0146s time_habr=0.0023s time_ols=0.0028s time_svd=0.0457s time_svd_knn=0.0340s time_kernel=8.1629s time_qda=0.0083s time_sk_qda=0.0088s

4: samples=9312, features=12, classes=5, acc=0.769 sk=0.768 habr=0.769 ols=0.766 svd=0.

## LDA через МНК (с разложением и без)

#### Уравнение множественной линейной регресии

$$
B = (X^T X)^{-1} X^T Y
$$

In [629]:
class OLS_LDA():
    def __init__(self, bias = False, svd = False, reg = None):
        self.svd = svd
        self.reg = reg
        self.bias = bias

    def one_hoter_y(self,y):   #pd.get_dummies(y)
        unique_classes = np.unique(y)
        n_unique = len(unique_classes)
        #n_samples, n_features = X.shape
        #zeros = np.zeros((n_samples, n_unique))
        labels = np.array([y==cls for cls in unique_classes]).T.astype(int)
        #columns = {cls : column for column, cls in zip(labels, unique_classes)}
        df = pd.DataFrame(labels, columns=unique_classes)
        return df

    def fit(self, X, y):
        
        n_samples, n_features = X.shape
        Y = self.one_hoter_y(y)
        self.columns = Y.columns
        Y = Y.to_numpy()

        if self.bias:
            X = np.hstack([np.ones((n_samples,1)), X])
            n_features = n_features + 1
            
        dot_product = X.T @ X
        if self.reg is not None:
            reg_num = np.trace(dot_product) * self.reg / n_features
            diag = reg_num * np.eye(n_features)
        else:
            diag = np.zeros((n_features, n_features))
            
        inv_matrix = np.linalg.inv(dot_product + diag) #np.linalg.pinv(dot_product)
        self.theta = inv_matrix @ X.T @ Y

        if self.svd:
            results = X @ self.theta
            U, S, Vt = np.linalg.svd(results, full_matrices=False)
            n_classes = len(self.columns)
            
            self.W = Vt.T[:, :n_classes-1]          # (n_classes, n_classes-1)
            self.projection = results @ self.W      # (n_samples, n_classes-1)
    
            self.proj_centers = []
            for cls in self.columns:
                mask = (y == cls)
                proj_cls = self.projection[mask].mean(axis=0)
                self.proj_centers.append(proj_cls)
            self.proj_centers = np.array(self.proj_centers)  # (n_classes, n_classes-1)

    def transform(self, X):
        results = X @ self.theta
        U, S, Vt = np.linalg.svd(results, full_matrices=False)
        n_classes = len(self.columns)
        return U[:, :n_classes-1]  # (n_samples, n_classes-1)

    def predict(self, X):
        if self.bias:
            n_samples, _ = X.shape
            X = np.hstack([np.ones((n_samples, 1)), X])
        classes = self.columns
        if not self.svd:
            results = X @ self.theta  # (n_samples, n_classes)
            pred_idxs = np.argmax(results, axis=1)
            predictions = np.take(classes, pred_idxs)
        else:
            results = X @ self.theta
            projected_X = results @ self.W

            dist = np.linalg.norm(
                projected_X[:, np.newaxis, :] -
                self.proj_centers[np.newaxis, :, :],
                axis=2)

            pred_idxs = np.argmin(dist, axis=1)
            return np.take(self.columns, pred_idxs)

        return predictions


In [637]:
ols = OLS_LDA(reg=1e-1, bias=False, svd=True)
ols.fit(X_train,y_train)
res = ols.predict(X_test)
print(accuracy_score(y_test, res))

0.6882882882882883


## LDA через SVD в реализации критерия Фишера

#### Замена T

$$
T = U_r \Sigma_r^{-1/2}
$$

In [676]:
import numpy as np


class LDA_SVD:

    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None
        self.classes_ = None
        self.means_ = None

    def fit(self, X, y):

        n_samples, n_features = X.shape

        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        mean_total = np.mean(X, axis=0)


        Sw = np.zeros((n_features, n_features))
        Sb = np.zeros((n_features, n_features))

        self.means_ = {}

        for cls in self.classes_:

            Xc = X[y == cls]

            mean_cls = np.mean(Xc, axis=0)
            self.means_[cls] = mean_cls

            X_centered = Xc - mean_cls
            Sw += X_centered.T @ X_centered

            diff = (mean_cls - mean_total).reshape(-1, 1)
            Sb += len(Xc) * diff @ diff.T

.
        U, singular_values, _ = np.linalg.svd(Sw)

        tol = 1e-10

        mask = singular_values > tol

        U_r = U[:, mask]
        Sigma_r = singular_values[mask]

        r = len(Sigma_r)

        Sigma_inv_sqrt = np.diag(1.0 / np.sqrt(Sigma_r))

        T = U_r @ Sigma_inv_sqrt
        A = T.T @ Sb @ T


        eigenvalues, eigenvectors = np.linalg.eigh(A)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        max_components = min(n_classes - 1, r)

        if self.n_components is None:
            n_components = max_components
        else:
            n_components = min(self.n_components, max_components)

        V = eigenvectors[:, :n_components]


        self.components_ = T @ V
      
        self.class_means_ = {}
        
        for cls in self.classes_:
            self.class_means_[cls] = self.means_[cls] @ self.components_
            
        return self

    def transform(self, X):
        return X @ self.components_

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

    def predict(self, X):

        Z = self.transform(X)
        predictions = []
    
        for z in Z:
            distances = []
            for cls in self.classes_:
                center = self.class_means_[cls]
                d = np.linalg.norm(z - center)
                distances.append(d)
            predictions.append(self.classes_[np.argmin(distances)])
        return np.array(predictions)

## Тест LDA_SVD как классификатор

In [677]:
lda = LDA_SVD(n_components=4)
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(acc)

0.6798516687268232


## Тест LDA_SVD как понижение размерности и метод knn

In [697]:
lda2 = LDA_SVD(n_components=7)
new_X_train = lda2.fit_transform(X_train, y_train)
new_X_test = lda2.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=30)
knn.fit(new_X_train, y_train)
y_pred_knn = knn.predict(new_X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(acc_knn)

0.7892459826946848


## LDA с ядерным трюком (Kernel LDA)

#### Kernel Trick

$$
v = \sum_{i=1}^n \alpha_i \phi(x_i)
$$$$
J(\alpha) = \frac{\alpha^T K W K \alpha}{\alpha^T K K \alpha}
$$

In [773]:
class KLDA():
    def __init__(self, kernel_func="linear", degree=2, gamma=1, reg=None):
        self.kernel = {
            'poly': lambda x, y: np.dot(x, y.T) ** degree,
            'rbf': lambda x, y: np.exp(-gamma * np.sum((y - x[:, np.newaxis]) ** 2, axis=-1)),
            'linear': lambda x, y: np.dot(x, y.T)
        }[kernel_func]
        self.reg = reg

    def _center_kernel(self, K):
        n = K.shape[0]
        one_n = np.ones((n, n)) / n
        return K - one_n @ K - K @ one_n + one_n @ K @ one_n

    def fit(self, X, y):
        self.X_train = X  
        self.y_train = y  
        
        n_samples, n_features = X.shape
        unique_classes, count_classes = np.unique(y, return_counts=True)
        n_classes = len(unique_classes)
 
        weights = np.array([1 / count_classes[np.where(unique_classes == cls)[0][0]] for cls in y])
        #diag_W = np.diag(weights)

        self.K = self._center_kernel(self.kernel(X, X))
        try:
            if self.reg is not None:
                reg = self.reg * np.trace(self.K) / n_samples
                inv_K = np.linalg.inv(self.K + reg * np.eye(n_samples))
            else:
                inv_K = np.linalg.inv(self.K)
        except:
            inv_K = np.linalg.pinv(self.K)
        
        #A = inv_K @ diag_W @ self.K
        A = inv_K @ (self.K * weights)
        evals, evecs = eigh(A)

        idx = np.argsort(evals.real)[::-1]
        self.alphas = evecs[:, idx[:n_classes - 1]].real  # (n_samples, n_components)

        projection_train = self.K @ self.alphas  # (n_samples, n_components)

        self.proj_centers = []
        for cls in unique_classes:
            mask = (y == cls)
            center = projection_train[mask].mean(axis=0)
            self.proj_centers.append(center)
        self.proj_centers = np.array(self.proj_centers)  # (n_classes, n_components)

    def predict(self, X):
        K_new = self.kernel(self.X_train, X)  # (n_samples, n_new_samples)

        projection_test = K_new.T @ self.alphas  # (n_new_samples, n_components)

        dist = np.linalg.norm(
            projection_test[:, np.newaxis, :] - self.proj_centers[np.newaxis, :, :],
            axis=2
        )  # (n_new_samples, n_classes)

        pred_idxs = np.argmin(dist, axis=1)  # (n_new_samples,)
        predictions = np.take(np.unique(self.y_train), pred_idxs)

        return predictions

In [774]:
klda = KLDA(reg=1e-5, kernel_func="linear")
klda.fit(X_train, y_train)
y_pred = klda.predict(X_test)
acc = accuracy_score(y_pred, y_test)
print(acc)

0.396168108776267


## Вероятностный QDA

#### Дискриминантная функция

$$\delta_y(x) = \log P(y) - \frac{1}{2}\log\Sigma_y- \frac{1}{2}(x - \mu_y)^T \Sigma_y^{-1} (x - \mu_y)$$

In [890]:
class ProbQDA():

    def __init__(self, reg = None, shrinkage = None):
        self.reg = reg
        self.lam = shrinkage
    
    def fit(self,X,y):
        n_samples, n_features = X.shape
        self.unique_classes, counts = np.unique(y, return_counts=True)
        self.priors = counts / n_samples
        
        self.classes_means = np.array([X[y==cls].mean(axis=0) for cls in self.unique_classes])

        self.cvr_matrices = []
        self.inv_cvr_matrices = []
        self.log_dets = []
        
        for idx, cls in enumerate(self.unique_classes):
            X_cls = X[y == cls]
            centered = X_cls - self.classes_means[idx]
            cov_cls = 1 / X_cls.shape[0] * centered.T @ centered  # (P x P)
            #shrinkaged_cov_cls = (1 - self.lam) * cov_cls + self.lam * np.eye(n_features) if self.lam is not None else cov_cls
            shrinked_cov_cls = ((1 - self.lam) * cov_cls + self.lam * np.trace(cov_cls) / n_features * np.eye(n_features))
            self.cvr_matrices.append(shrinked_cov_cls)

            try:
                if self.reg is not None:
                    reg = self.reg * np.trace(shrinked_cov_cls) / n_features
                    inv_cov = np.linalg.inv(shrinked_cov_cls + reg * np.eye(n_features))
                else:
                    inv_cov = np.linalg.inv(shrinked_cov_cls)
            except:
                inv_cov = np.linalg.pinv(shrinked_cov_cls)
            self.inv_cvr_matrices.append(inv_cov)
            
            sign, logdet = np.linalg.slogdet(shrinked_cov_cls)  # логарифмы определителей ковариационных матриц
            self.log_dets.append(logdet)
        
        self.inv_cvr_matrices = np.array(self.inv_cvr_matrices)
        self.log_dets = np.array(self.log_dets)
        
        self.const = np.array([
            np.log(self.priors[k]) 
            - 0.5 * self.log_dets[k] 
            - 0.5 * self.classes_means[k] @ self.inv_cvr_matrices[k] @ self.classes_means[k]
            for k in range(len(self.unique_classes))])

    def predict(self, X):
        X = np.asarray(X)
        n_samples = X.shape[0]
        n_classes = len(self.unique_classes)
        scores = np.zeros((n_samples, n_classes))
        
        for k in range(n_classes):
            inv_cov = self.inv_cvr_matrices[k]
            mean = self.classes_means[k]

            diff = X - mean  # (n_samples, n_features)
            #quadratic = np.sum((diff @ inv_cov) * diff, axis=1)
            quadratic = np.einsum('ij,jk,ik->i', diff, inv_cov, diff)  # (n_samples,)
            scores[:, k] = self.const[k] - 0.5 * quadratic
        
        max_idxs = np.argmax(scores, axis=1)
        predictions = np.take(self.unique_classes, max_idxs)

        return predictions

In [903]:
# QDA
qda = ProbQDA(shrinkage=0.15, reg = 1e-12)
qda.fit(X_train, y_train)
print(f"QDA accuracy: {accuracy_score(y_test, qda.predict(X_test)):.3f}")

QDA accuracy: 0.927
